---
# 🐼 Pandas Practice Lab — E-Commerce Edition
### A self-paced exercise notebook for data practitioners

**Built by:** Afnan Faiyazahmed Darga  
**Level:** Beginner → Intermediate → Advanced  
**Estimated time:** 60–90 minutes

---

## How this works

- Each section has **exercises** with a `# YOUR CODE HERE` cell
- Run the **check cell** right after — it prints `✅ CORRECT!` or tells you what's wrong
- Stuck? Scroll to the **💡 Hint** or reveal the **🔑 Solution** (inside the `<details>` block)
- **Do not use explicit `for` or `while` loops** unless the question asks for it

---

## The Dataset

You're working as a data analyst at **ShopStream**, a fictional e-commerce platform.
You have access to **orders data** from 2023: 1,000 transactions across products, customers, and regions.

**Columns:**

| Column | Description |
|---|---|
| `order_id` | Unique order identifier |
| `customer_id` | Customer ID (some customers have multiple orders) |
| `order_date` | Date the order was placed |
| `product` | Product name |
| `category` | Product category |
| `quantity` | Units ordered |
| `unit_price` | Price per unit (£) |
| `discount` | Discount applied (0.0–0.3, or NaN if no discount recorded) |
| `region` | UK region |
| `status` | Order status |

---

## 0 — Setup
**Just run this cell. Do not edit it.**

In [1]:
import pandas as pd
import numpy as np

# Dataset generation (fixed seed -- same data every run)
rng = np.random.default_rng(42)
N   = 1000

products = [
    ("Wireless Earbuds",    "Electronics"),
    ("Laptop Stand",        "Electronics"),
    ("USB-C Hub",           "Electronics"),
    ("Mechanical Keyboard", "Electronics"),
    ("Yoga Mat",            "Fitness"),
    ("Resistance Bands",    "Fitness"),
    ("Foam Roller",         "Fitness"),
    ("Running Shoes",       "Fitness"),
    ("Coffee Maker",        "Kitchen"),
    ("Air Fryer",           "Kitchen"),
    ("Blender",             "Kitchen"),
    ("Desk Lamp",           "Home"),
    ("Scented Candle",      "Home"),
    ("Bookshelf",           "Home"),
    ("Python Book",         "Books"),
    ("Data Science Book",   "Books"),
]
unit_prices = {
    "Wireless Earbuds": 49.99, "Laptop Stand": 34.99, "USB-C Hub": 24.99,
    "Mechanical Keyboard": 89.99, "Yoga Mat": 29.99, "Resistance Bands": 14.99,
    "Foam Roller": 19.99, "Running Shoes": 74.99, "Coffee Maker": 59.99,
    "Air Fryer": 99.99, "Blender": 44.99, "Desk Lamp": 27.99,
    "Scented Candle": 9.99, "Bookshelf": 79.99,
    "Python Book": 29.99, "Data Science Book": 34.99,
}
regions  = ["London", "Manchester", "Birmingham", "Edinburgh", "Bristol", "Leeds"]
statuses = ["Delivered", "Delivered", "Delivered", "Shipped", "Processing", "Cancelled"]

prod_idx   = rng.integers(0, len(products), N)
prod_names = [products[i][0] for i in prod_idx]
categories = [products[i][1] for i in prod_idx]

dates = pd.date_range("2023-01-01", "2023-12-31", periods=N)
dates = dates[rng.choice(len(dates), N, replace=False)].sort_values()

discounts_raw = rng.choice(
    [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, np.nan], N,
    p=[0.10, 0.12, 0.10, 0.08, 0.05, 0.05, 0.50])

orders = pd.DataFrame({
    "order_id":    [f"ORD{str(i).zfill(5)}" for i in range(1, N + 1)],
    "customer_id": [f"CUST{str(rng.integers(1, 201)).zfill(4)}" for _ in range(N)],
    "order_date":  dates,
    "product":     prod_names,
    "category":    categories,
    "quantity":    rng.integers(1, 6, N),
    "unit_price":  [unit_prices[p] for p in prod_names],
    "discount":    discounts_raw.astype(float),
    "region":      rng.choice(regions, N),
    "status":      rng.choice(statuses, N, p=[0.55, 0.15, 0.10, 0.10, 0.05, 0.05]),
})

# Score tracker
_score = {"correct": 0, "total": 0}

def check(result, expected, label=""):
    """Auto-checker: prints CORRECT or Not quite. Never crashes."""
    tag = f" [{label}]" if label else ""
    _score["total"] += 1

    # Guard: unattempted answer
    if result is None:
        print(f"Not quite.{tag} Got: None -- write your answer above and remove the raise line.")
        return

    try:
        if isinstance(expected, float):
            ok = abs(float(result) - expected) < 0.01
        elif isinstance(expected, (int, np.integer)):
            ok = int(result) == int(expected)
        elif isinstance(expected, str):
            ok = str(result).strip() == str(expected).strip()
        elif isinstance(expected, (list, np.ndarray)):
            try:
                ok = np.allclose(
                    np.array(result, dtype=float),
                    np.array(expected, dtype=float), atol=0.01)
            except (ValueError, TypeError):
                ok = list(result) == list(expected)
        elif isinstance(expected, pd.Index):
            ok = list(result) == list(expected)
        else:
            ok = result == expected
    except Exception:
        ok = False

    if ok:
        _score["correct"] += 1
        print(f"CORRECT!{tag}")
    else:
        print(f"Not quite.{tag} Got: {result!r}  |  Expected: {expected!r}")

print("Dataset ready! Shape:", orders.shape)
print("Columns:", list(orders.columns))


Dataset ready! Shape: (1000, 10)
Columns: ['order_id', 'customer_id', 'order_date', 'product', 'category', 'quantity', 'unit_price', 'discount', 'region', 'status']


---
# Section 1 — Exploring Your Data 🟢
*Beginner — get familiar with the dataset*

### Exercise 1.1 — First look
Display the **first 5 rows** of the `orders` DataFrame.

In [2]:
# YOUR CODE HERE
# raise NotImplementedError()
orders.head()

,order_id,customer_id,order_date,product,category,quantity,unit_price,discount,region,status
0,ORD00001,CUST0136,2023-01-01 00:00:00.000000000,Laptop Stand,Electronics,4,34.99,NaN,Birmingham,Delivered
1,ORD00002,CUST0100,2023-01-01 08:44:41.081081081,Scented Candle,Home,1,9.99,NaN,Leeds,Delivered
2,ORD00003,CUST0040,2023-01-01 17:29:22.162162162,Blender,Kitchen,4,44.99,NaN,Leeds,Delivered
3,ORD00004,CUST0007,2023-01-02 02:14:03.243243243,Running Shoes,Fitness,5,74.99,0.20,Birmingham,Shipped
4,ORD00005,CUST0126,2023-01-02 10:58:44.324324324,Foam Roller,Fitness,4,19.99,0.25,Leeds,Delivered


<details><summary>🔑 Solution</summary>

```python
orders.head()
```
</details>

### Exercise 1.2 — How big is this dataset?
Store the number of **rows** in `n_rows` and the number of **columns** in `n_cols`.

In [3]:
n_rows = None
n_cols = None
# YOUR CODE HERE
# raise NotImplementedError()
n_rows, n_cols = orders.shape


In [4]:
check(n_rows, 1000, "n_rows")
check(n_cols, 10,   "n_cols")


CORRECT! [n_rows]
CORRECT! [n_cols]


<details><summary>🔑 Solution</summary>

```python
n_rows, n_cols = orders.shape
```
</details>

### Exercise 1.3 — Missing values
Store the **total number of missing values** across the whole DataFrame in `total_missing`.

In [5]:
total_missing = None
# YOUR CODE HERE
# raise NotImplementedError()
total_missing = orders.isnull().sum().sum()

In [6]:
check(total_missing, int(orders.isnull().sum().sum()), "total_missing")


CORRECT! [total_missing]


<details><summary>💡 Hint</summary>
Use `.isnull()` then `.sum()` — you'll need to call `.sum()` twice (once per axis, once across the result).
</details>

<details><summary>🔑 Solution</summary>

```python
total_missing = orders.isnull().sum().sum()
```
</details>

### Exercise 1.4 — Unique categories
Store the list of **unique product categories** (sorted A→Z) in `categories`.

Expected: `['Books', 'Electronics', 'Fitness', 'Home', 'Kitchen']`

In [7]:
categories = None
# YOUR CODE HERE
# raise NotImplementedError()
categories = sorted(orders['category'].unique())

In [8]:
assert categories is not None, "categories is still None -- write your answer and remove the raise line"
expected = ['Books', 'Electronics', 'Fitness', 'Home', 'Kitchen']
assert list(sorted(categories)) == expected, f"Got: {sorted(categories)}"
print("CORRECT! [categories]")


CORRECT! [categories]


<details><summary>🔑 Solution</summary>

```python
categories = sorted(orders['category'].unique())
```
</details>

### Exercise 1.5 — Most common region
Store the **region with the most orders** in `top_region`.

In [9]:
top_region = None
# YOUR CODE HERE
# raise NotImplementedError()
top_region = orders['region'].value_counts().idxmax()

In [10]:
check(top_region, orders['region'].value_counts().idxmax(), "top_region")


CORRECT! [top_region]


<details><summary>💡 Hint</summary>
`.value_counts()` counts occurrences. Use `.idxmax()` to get the label of the highest count.
</details>

<details><summary>🔑 Solution</summary>

```python
top_region = orders['region'].value_counts().idxmax()
```
</details>

---
# Section 2 — Indexing & Slicing 🟢
*Beginner — `loc`, `iloc`, and column selection*

### Exercise 2.1 — Select specific columns
Create a DataFrame `order_summary` containing only the columns: `order_id`, `product`, `quantity`, `unit_price`.

In [11]:
order_summary = None
# YOUR CODE HERE
# raise NotImplementedError()
order_summary = orders[['order_id', 'product', 'quantity', 'unit_price']]

In [12]:
assert order_summary is not None, "order_summary is still None -- write your answer and remove the raise line"
assert list(order_summary.columns) == ['order_id', 'product', 'quantity', 'unit_price'], "Wrong columns"
assert len(order_summary) == 1000, "Wrong number of rows"
print("CORRECT! [order_summary]")


CORRECT! [order_summary]


<details><summary>🔑 Solution</summary>

```python
order_summary = orders[['order_id', 'product', 'quantity', 'unit_price']]
```
</details>

### Exercise 2.2 — Row by position
Use `iloc` to extract the **100th row** (0-indexed → row at position 99) and store it in `row_100`.

In [13]:
row_100 = None
# YOUR CODE HERE
# raise NotImplementedError()
row_100 = orders.iloc[99]

In [14]:
assert row_100 is not None, "row_100 is still None -- write your answer and remove the raise line"
assert row_100['order_id'] == orders.iloc[99]['order_id'], "Wrong row"
print("CORRECT! [row_100]")
print(row_100)


CORRECT! [row_100]
order_id                            ORD00100
customer_id                         CUST0064
order_date     2023-02-06 01:43:47.027027027
product                            USB-C Hub
category                         Electronics
quantity                                   5
unit_price                             24.99
discount                                 NaN
region                                London
status                             Delivered
Name: 99, dtype: object


<details><summary>🔑 Solution</summary>

```python
row_100 = orders.iloc[99]
```
</details>

### Exercise 2.3 — Slice a range
Use `iloc` to extract rows **200 to 209** (inclusive) and store in `slice_200`.

In [15]:
slice_200 = None
# YOUR CODE HERE
# raise NotImplementedError()
slice_200 = orders.iloc[200:210]

In [16]:
assert slice_200 is not None, "slice_200 is still None -- write your answer and remove the raise line"
assert len(slice_200) == 10, f"Expected 10 rows, got {len(slice_200)}"
assert list(slice_200.index) == list(orders.iloc[200:210].index), "Wrong rows selected"
print("CORRECT! [slice_200]")


CORRECT! [slice_200]


<details><summary>🔑 Solution</summary>

```python
slice_200 = orders.iloc[200:210]
```
</details>

### Exercise 2.4 — Lookup by label
Use `loc` to get the **unit_price of the order at index 0** and store it in `first_price`.

In [17]:
first_price = None
# YOUR CODE HERE
# raise NotImplementedError()
first_price = orders.loc[0, 'unit_price']

In [18]:
check(first_price, orders.loc[0, 'unit_price'], "first_price")


CORRECT! [first_price]


<details><summary>🔑 Solution</summary>

```python
first_price = orders.loc[0, 'unit_price']
```
</details>

---
# Section 3 — Filtering & Sorting 🟢🟠
*Beginner-Intermediate — boolean masks and ordering*

### Exercise 3.1 — Filter by category
Store all **Electronics orders** in `electronics_orders`.

In [19]:
electronics_orders = None
# YOUR CODE HERE
# raise NotImplementedError()
electronics_orders = orders[orders['category'] == 'Electronics']


In [20]:
assert electronics_orders is not None, "electronics_orders is still None -- write your answer and remove the raise line"
expected = orders[orders['category'] == 'Electronics']
assert len(electronics_orders) == len(expected), f"Expected {len(expected)} rows, got {len(electronics_orders)}"
assert (electronics_orders['category'] == 'Electronics').all(), "Non-electronics rows found"
print(f"CORRECT! [electronics_orders] -- {len(electronics_orders)} rows")


CORRECT! [electronics_orders] -- 253 rows


<details><summary>🔑 Solution</summary>

```python
electronics_orders = orders[orders['category'] == 'Electronics']
```
</details>

### Exercise 3.2 — Multi-condition filter
Store orders that are **Delivered** AND have a **unit_price above £50** in `high_value_delivered`.

In [21]:
high_value_delivered = None
# YOUR CODE HERE
# raise NotImplementedError()
high_value_delivered = orders[(orders['status'] == 'Delivered') & (orders['unit_price'] > 50)]


In [22]:
assert high_value_delivered is not None, "high_value_delivered is still None -- write your answer and remove the raise line"
expected = orders[(orders['status'] == 'Delivered') & (orders['unit_price'] > 50)]
assert len(high_value_delivered) == len(expected), f"Expected {len(expected)} rows, got {len(high_value_delivered)}"
print(f"CORRECT! [high_value_delivered] -- {len(high_value_delivered)} rows")


CORRECT! [high_value_delivered] -- 251 rows


<details><summary>💡 Hint</summary>
Use `&` to combine conditions. Always wrap each condition in parentheses: `(cond1) & (cond2)`.
</details>

<details><summary>🔑 Solution</summary>

```python
high_value_delivered = orders[(orders['status'] == 'Delivered') & (orders['unit_price'] > 50)]
```
</details>

### Exercise 3.3 — Filter using `.isin()`
Store orders from **London** or **Edinburgh** in `london_edinburgh`.

In [23]:
london_edinburgh = None
# YOUR CODE HERE
# raise NotImplementedError()
london_edinburgh = orders[orders['region'].isin(['London', 'Edinburgh'])]

In [24]:
assert london_edinburgh is not None, "london_edinburgh is still None -- write your answer and remove the raise line"
expected = orders[orders['region'].isin(['London', 'Edinburgh'])]
assert len(london_edinburgh) == len(expected), f"Expected {len(expected)} rows, got {len(london_edinburgh)}"
print(f"CORRECT! [london_edinburgh] -- {len(london_edinburgh)} rows")


CORRECT! [london_edinburgh] -- 342 rows


<details><summary>🔑 Solution</summary>

```python
london_edinburgh = orders[orders['region'].isin(['London', 'Edinburgh'])]
```
</details>

### Exercise 3.4 — Sort by value
Create `top10_expensive` — the **10 most expensive** orders by `unit_price`, highest first. Keep all columns.

In [25]:
top10_expensive = None
# YOUR CODE HERE
# raise NotImplementedError()

top10_expensive = orders.sort_values('unit_price', ascending=False).head(10)

In [26]:
assert top10_expensive is not None, "top10_expensive is still None -- write your answer and remove the raise line"
assert len(top10_expensive) == 10, "Expected 10 rows"
assert top10_expensive['unit_price'].is_monotonic_decreasing, "Not sorted highest first"
print("CORRECT! [top10_expensive]")
print(top10_expensive[['product', 'unit_price']].to_string())


CORRECT! [top10_expensive]
       product  unit_price
914  Air Fryer       99.99
671  Air Fryer       99.99
241  Air Fryer       99.99
519  Air Fryer       99.99
596  Air Fryer       99.99
436  Air Fryer       99.99
707  Air Fryer       99.99
705  Air Fryer       99.99
701  Air Fryer       99.99
929  Air Fryer       99.99


<details><summary>🔑 Solution</summary>

```python
top10_expensive = orders.sort_values('unit_price', ascending=False).head(10)
```
</details>

### Exercise 3.5 — Filter with `.query()`
Use `.query()` to get all **Fitness orders with quantity >= 3**. Store in `fitness_bulk`.

*Note: `.query()` is a clean alternative to boolean indexing — worth knowing!*

In [27]:
fitness_bulk = None
# YOUR CODE HERE
# raise NotImplementedError()
fitness_bulk=orders.query("category == 'Fitness' and quantity >= 3")

In [28]:
assert fitness_bulk is not None, "fitness_bulk is still None -- write your answer and remove the raise line"
expected = orders[(orders['category'] == 'Fitness') & (orders['quantity'] >= 3)]
assert len(fitness_bulk) == len(expected), f"Expected {len(expected)} rows, got {len(fitness_bulk)}"
print(f"CORRECT! [fitness_bulk] -- {len(fitness_bulk)} rows")


CORRECT! [fitness_bulk] -- 130 rows


<details><summary>💡 Hint</summary>

```python
orders.query("category == 'Fitness' and quantity >= 3")
```
</details>

---
# Section 4 — Derived Columns & Missing Data 🟠
*Intermediate — adding computed columns, handling NaN*

### Exercise 4.1 — Compute revenue
Add a new column `revenue` to `orders`:  
`revenue = quantity × unit_price`

*(Ignore discounts for now — we'll apply them in 4.2)*

In [29]:
# YOUR CODE HERE
# raise NotImplementedError()
orders['revenue'] = orders['quantity'] * orders['unit_price']

In [30]:
assert 'revenue' in orders.columns, "Column 'revenue' not found -- write your answer and remove the raise line"
expected_first = orders.iloc[0]['quantity'] * orders.iloc[0]['unit_price']
check(orders.iloc[0]['revenue'], expected_first, "revenue[0]")


CORRECT! [revenue[0]]


<details><summary>🔑 Solution</summary>

```python
orders['revenue'] = orders['quantity'] * orders['unit_price']
```
</details>

### Exercise 4.2 — Apply discounts
Add a column `final_revenue`:  
`final_revenue = revenue × (1 − discount)`  

Where `discount` is NaN, treat it as **0** (no discount applied).

In [31]:
# YOUR CODE HERE
# raise NotImplementedError()
orders['final_revenue'] = orders['revenue'] * (1 - orders['discount'].fillna(0))

In [32]:
assert 'revenue' in orders.columns, "Solve Exercise 4.1 first -- 'revenue' column is missing"
assert 'final_revenue' in orders.columns, "Column 'final_revenue' not found -- write your answer and remove the raise line"
assert orders['final_revenue'].isnull().sum() == 0, "final_revenue should have no NaN values"
discount_rows = orders[orders['discount'].notna()]
r = discount_rows.iloc[0]
expected_val = r['revenue'] * (1 - r['discount'])
check(r['final_revenue'], expected_val, "final_revenue[discounted row]")


CORRECT! [final_revenue[discounted row]]


<details><summary>💡 Hint</summary>
Use `.fillna(0)` on the `discount` column before computing.
</details>

<details><summary>🔑 Solution</summary>

```python
orders['final_revenue'] = orders['revenue'] * (1 - orders['discount'].fillna(0))
```
</details>

### Exercise 4.3 — Flag discounted orders
Add a boolean column `has_discount` that is `True` if a discount was applied, `False` otherwise.

In [33]:
# YOUR CODE HERE
# raise NotImplementedError()
orders['has_discount'] = orders['discount'].notna()

In [34]:
assert 'final_revenue' in orders.columns, "Solve Exercise 4.2 first -- 'final_revenue' column is missing"
assert 'has_discount' in orders.columns, "Column 'has_discount' not found -- write your answer and remove the raise line"
assert orders['has_discount'].dtype == bool, "has_discount should be boolean dtype"
check(orders['has_discount'].sum(), orders['discount'].notna().sum(), "count of discounted orders")


CORRECT! [count of discounted orders]


<details><summary>🔑 Solution</summary>

```python
orders['has_discount'] = orders['discount'].notna()
```
</details>

---
# Section 5 — GroupBy & Aggregations 🟠
*Intermediate — the heart of Pandas analysis*

### Exercise 5.1 — Total revenue by category
Compute the **total `final_revenue` per category**. Store as a Series `revenue_by_category`, sorted highest first.

In [35]:
revenue_by_category = None
# YOUR CODE HERE
# raise NotImplementedError()
revenue_by_category = orders.groupby('category')['final_revenue'].sum().sort_values(ascending=False)

In [36]:
assert 'final_revenue' in orders.columns, "Solve Exercise 4.2 first -- 'final_revenue' column is missing"
assert revenue_by_category is not None, "revenue_by_category is still None -- write your answer and remove the raise line"
expected = orders.groupby('category')['final_revenue'].sum().sort_values(ascending=False)
assert list(revenue_by_category.index) == list(expected.index), "Wrong order or categories"
assert np.allclose(revenue_by_category.values, expected.values, atol=0.01), "Wrong values"
print("CORRECT! [revenue_by_category]")
print(revenue_by_category.round(2))


CORRECT! [revenue_by_category]
category
Kitchen        36792.76
Electronics    31345.25
Fitness        23628.59
Home           18756.70
Books           9761.75
Name: final_revenue, dtype: float64


<details><summary>🔑 Solution</summary>

```python
revenue_by_category = orders.groupby('category')['final_revenue'].sum().sort_values(ascending=False)
```
</details>

### Exercise 5.2 — Order count by status
Store the **number of orders per status** in `orders_by_status`.

In [37]:
orders_by_status = None
# YOUR CODE HERE
# raise NotImplementedError()
orders_by_status = orders.groupby('status')['order_id'].count()
# Alternative: orders['status'].value_counts()

In [38]:
assert orders_by_status is not None, "orders_by_status is still None -- write your answer and remove the raise line"
expected = orders.groupby('status')['order_id'].count()
assert set(orders_by_status.index) == set(expected.index), "Wrong status labels"
assert orders_by_status.sum() == 1000, "Counts don't add up to 1000"
print("CORRECT! [orders_by_status]")
print(orders_by_status)


CORRECT! [orders_by_status]
status
Cancelled      38
Delivered     820
Processing     46
Shipped        96
Name: order_id, dtype: int64


<details><summary>🔑 Solution</summary>

```python
orders_by_status = orders.groupby('status')['order_id'].count()
# Alternative: orders['status'].value_counts()
```
</details>

### Exercise 5.3 — Multi-metric aggregation with `.agg()`
For each **region**, compute:
- `total_revenue`: sum of `final_revenue`
- `avg_order_value`: mean of `final_revenue`
- `order_count`: count of `order_id`

Store as DataFrame `region_stats`.

In [39]:
region_stats = None
# YOUR CODE HERE
# raise NotImplementedError()
region_stats = orders.groupby('region').agg(
    total_revenue   = ('final_revenue', 'sum'),
    avg_order_value = ('final_revenue', 'mean'),
    order_count     = ('order_id', 'count')
)

In [40]:
assert 'final_revenue' in orders.columns, "Solve Exercise 4.2 first -- 'final_revenue' column is missing"
assert isinstance(region_stats, pd.DataFrame), "region_stats should be a DataFrame -- write your answer and remove the raise line"
assert set(region_stats.columns) == {'total_revenue', 'avg_order_value', 'order_count'}, f"Wrong columns: {list(region_stats.columns)}"
assert len(region_stats) == 6, "Should have 6 regions"
print("CORRECT! [region_stats]")
print(region_stats.round(2))


CORRECT! [region_stats]
            total_revenue  avg_order_value  order_count
region                                                 
Birmingham       20621.54           122.02          169
Bristol          19621.21           124.18          158
Edinburgh        20909.49           119.48          175
Leeds            20066.92           126.21          159
London           19710.03           118.02          167
Manchester       19355.85           112.53          172


<details><summary>💡 Hint</summary>
Use `.agg()` with a dictionary: `{'col': 'func', ...}` or named aggregations.
</details>

<details><summary>🔑 Solution</summary>

```python
region_stats = orders.groupby('region').agg(
    total_revenue   = ('final_revenue', 'sum'),
    avg_order_value = ('final_revenue', 'mean'),
    order_count     = ('order_id', 'count')
)
```
</details>

### Exercise 5.4 — Best-selling product
Find the **product with the highest total quantity sold** and store it in `best_seller`.

In [41]:
best_seller = None
# YOUR CODE HERE
# raise NotImplementedError()
best_seller = orders.groupby('product')['quantity'].sum().idxmax()

In [42]:
expected = orders.groupby('product')['quantity'].sum().idxmax()
check(best_seller, expected, "best_seller")


CORRECT! [best_seller]


<details><summary>🔑 Solution</summary>

```python
best_seller = orders.groupby('product')['quantity'].sum().idxmax()
```
</details>

### Exercise 5.5 — Top customer by spend
Find the **customer_id that spent the most** (highest total `final_revenue`). Store in `top_customer`.

In [43]:
top_customer = None
# YOUR CODE HERE
# raise NotImplementedError()
top_customer = orders.groupby('customer_id')['final_revenue'].sum().idxmax()

In [44]:
assert 'final_revenue' in orders.columns, "Solve Exercise 4.2 first -- 'final_revenue' column is missing"
expected = orders.groupby('customer_id')['final_revenue'].sum().idxmax()
check(top_customer, expected, "top_customer")


CORRECT! [top_customer]


<details><summary>🔑 Solution</summary>

```python
top_customer = orders.groupby('customer_id')['final_revenue'].sum().idxmax()
```
</details>

---
# Section 6 — DateTime & String Operations 🟠
*Intermediate — working with dates and text columns*

### Exercise 6.1 — Extract the month
Add a column `month` (integer, 1–12) to `orders` using the `order_date` column.

In [45]:
# YOUR CODE HERE
# raise NotImplementedError()
orders['month'] = orders['order_date'].dt.month

In [46]:
assert 'month' in orders.columns, "Column 'month' not found -- write your answer and remove the raise line"
assert orders['month'].between(1, 12).all(), "Month values should be between 1 and 12"
assert orders['month'].dtype in [int, np.int32, np.int64], "month should be integer type"
print("CORRECT! [month]")
print(orders['month'].value_counts().sort_index())


CORRECT! [month]
month
1     86
2     76
3     86
4     82
5     85
6     82
7     85
8     85
9     83
10    85
11    82
12    83
Name: count, dtype: int64


<details><summary>💡 Hint</summary>
Use the `.dt` accessor: `orders['order_date'].dt.month`
</details>

<details><summary>🔑 Solution</summary>

```python
orders['month'] = orders['order_date'].dt.month
```
</details>

### Exercise 6.2 — Monthly revenue trend
Compute total `final_revenue` per month. Store as Series `monthly_revenue` (index = month number, sorted 1→12).

In [47]:
monthly_revenue = None
# YOUR CODE HERE
# raise NotImplementedError()
monthly_revenue = orders.groupby('month')['final_revenue'].sum().sort_index()

In [48]:
assert 'month' in orders.columns, "Solve Exercise 6.1 first -- 'month' column is missing"
assert 'final_revenue' in orders.columns, "Solve Exercise 4.2 first -- 'final_revenue' column is missing"
assert monthly_revenue is not None, "monthly_revenue is still None -- write your answer and remove the raise line"
expected = orders.groupby('month')['final_revenue'].sum().sort_index()
assert len(monthly_revenue) == 12, "Should have 12 months"
assert np.allclose(monthly_revenue.values, expected.values, atol=0.01), "Revenue values don't match"
print("CORRECT! [monthly_revenue]")
print(monthly_revenue.round(2))


CORRECT! [monthly_revenue]
month
1      9183.15
2      9765.96
3      9443.46
4      9598.77
5      9259.96
6      9347.06
7     10195.61
8     10750.89
9      9035.47
10    11474.33
11    10855.02
12    11375.35
Name: final_revenue, dtype: float64


<details><summary>🔑 Solution</summary>

```python
monthly_revenue = orders.groupby('month')['final_revenue'].sum().sort_index()
```
</details>

### Exercise 6.3 — String search
Store all orders where the **product name contains the word "Book"** in `book_orders`.

In [49]:
book_orders = None
# YOUR CODE HERE
# raise NotImplementedError()
book_orders = orders[orders['product'].str.contains('Book', case=False)]

In [50]:
assert book_orders is not None, "book_orders is still None -- write your answer and remove the raise line"
expected = orders[orders['product'].str.contains('Book', case=False)]
assert len(book_orders) == len(expected), f"Expected {len(expected)} rows, got {len(book_orders)}"
print(f"CORRECT! [book_orders] -- {len(book_orders)} rows")


CORRECT! [book_orders] -- 181 rows


<details><summary>🔑 Solution</summary>

```python
book_orders = orders[orders['product'].str.contains('Book', case=False)]
```
</details>

### Exercise 6.4 — Extract order number
Add a column `order_num` that contains the **numeric part** of `order_id` as an integer.  
e.g. `"ORD00042"` → `42`

In [51]:
# YOUR CODE HERE
# raise NotImplementedError()
orders['order_num'] = orders['order_id'].str.replace('ORD', '').astype(int)
# or with regex:
# orders['order_num'] = orders['order_id'].str.extract(r'(\d+)').astype(int)

In [52]:
assert 'order_num' in orders.columns, "Column 'order_num' not found -- write your answer and remove the raise line"
assert orders['order_num'].iloc[0] == 1, f"First order_num should be 1, got {orders['order_num'].iloc[0]}"
assert orders['order_num'].iloc[-1] == 1000, f"Last order_num should be 1000, got {orders['order_num'].iloc[-1]}"
print("CORRECT! [order_num]")


CORRECT! [order_num]


<details><summary>💡 Hint</summary>
Use `.str.replace()` or `.str.extract()` with a regex pattern, then cast to int.
</details>

<details><summary>🔑 Solution</summary>

```python
orders['order_num'] = orders['order_id'].str.replace('ORD', '').astype(int)
# or with regex:
# orders['order_num'] = orders['order_id'].str.extract(r'(\d+)').astype(int)
```
</details>

---
# Section 7 — Merging DataFrames 🟠🔴
*Intermediate-Advanced — joining tables*

In [53]:
# Run this cell to create two additional tables to work with

# Customer info table
customer_ids = orders['customer_id'].unique()
tiers = rng.choice(['Bronze', 'Silver', 'Gold', 'Platinum'], len(customer_ids), p=[0.4, 0.3, 0.2, 0.1])
customers = pd.DataFrame({
    'customer_id':   customer_ids,
    'loyalty_tier':  tiers,
    'signup_year':   rng.integers(2018, 2024, len(customer_ids)),
})

# Returns table — some orders were returned
return_ids = rng.choice(orders['order_id'], 80, replace=False)
returns = pd.DataFrame({
    'order_id':      return_ids,
    'return_reason': rng.choice(['Defective', 'Wrong item', 'Changed mind', 'Late delivery'], 80),
})

print("customers table:", customers.shape)
print(customers.head(3))
print("\nreturns table:", returns.shape)
print(returns.head(3))

customers table: (199, 3)
  customer_id loyalty_tier  signup_year
0    CUST0136       Silver         2022
1    CUST0100       Bronze         2022
2    CUST0040       Silver         2019

returns table: (80, 2)
   order_id return_reason
0  ORD00721  Changed mind
1  ORD00036  Changed mind
2  ORD00735     Defective


### Exercise 7.1 — Inner join
Merge `orders` with `customers` on `customer_id` (inner join). Store in `orders_with_tier`.

In [54]:
orders_with_tier = None
# YOUR CODE HERE
# raise NotImplementedError()
orders_with_tier = orders.merge(customers, on='customer_id', how='inner')

In [55]:
assert orders_with_tier is not None, "orders_with_tier is still None -- write your answer and remove the raise line"
assert 'loyalty_tier' in orders_with_tier.columns, "loyalty_tier column missing"
assert 'signup_year'  in orders_with_tier.columns, "signup_year column missing"
assert len(orders_with_tier) == 1000, f"Expected 1000 rows after merge, got {len(orders_with_tier)}"
print("CORRECT! [orders_with_tier]")
print(orders_with_tier[['order_id', 'customer_id', 'loyalty_tier']].head())


CORRECT! [orders_with_tier]
   order_id customer_id loyalty_tier
0  ORD00001    CUST0136       Silver
1  ORD00002    CUST0100       Bronze
2  ORD00003    CUST0040       Silver
3  ORD00004    CUST0007         Gold
4  ORD00005    CUST0126       Bronze


<details><summary>🔑 Solution</summary>

```python
orders_with_tier = orders.merge(customers, on='customer_id', how='inner')
```
</details>

### Exercise 7.2 — Left join (find returned orders)
Merge `orders` with `returns` on `order_id` using a **left join**.  
Add a boolean column `was_returned` that is `True` if the order appears in `returns`.  
Store result in `orders_with_returns`.

In [56]:
orders_with_returns = None
# YOUR CODE HERE
# raise NotImplementedError()
orders_with_returns = orders.merge(returns, on='order_id', how='left')
orders_with_returns['was_returned'] = orders_with_returns['return_reason'].notna()

In [57]:
assert orders_with_returns is not None, "orders_with_returns is still None -- write your answer and remove the raise line"
assert len(orders_with_returns) == 1000, f"Left join should preserve all 1000 orders, got {len(orders_with_returns)}"
assert 'was_returned' in orders_with_returns.columns, "was_returned column missing -- did you add the boolean column?"
check(orders_with_returns['was_returned'].sum(), 80, "number of returned orders")


CORRECT! [number of returned orders]


<details><summary>💡 Hint</summary>
After the left join, `return_reason` will be NaN for non-returned orders. Use `.notna()` to create the boolean flag.
</details>

<details><summary>🔑 Solution</summary>

```python
orders_with_returns = orders.merge(returns, on='order_id', how='left')
orders_with_returns['was_returned'] = orders_with_returns['return_reason'].notna()
```
</details>

### Exercise 7.3 — Return rate by category
Using `orders_with_returns`, compute the **return rate per category** (% of orders returned).  
Store as Series `return_rate_by_category` with values as percentages (0–100), sorted highest first.

In [58]:
return_rate_by_category = None
# YOUR CODE HERE
# raise NotImplementedError()
return_rate_by_category = (
    orders_with_returns.groupby('category')['was_returned'].mean() * 100
).sort_values(ascending=False)

In [59]:
try:
    orders_with_returns
except NameError:
    raise AssertionError("Solve Exercise 7.2 first -- run cell 7.2 to initialise orders_with_returns")
assert orders_with_returns is not None, "Solve Exercise 7.2 first -- orders_with_returns is None"
assert 'was_returned' in orders_with_returns.columns, "Solve Exercise 7.2 first -- 'was_returned' column is missing"
assert return_rate_by_category is not None, "return_rate_by_category is still None -- write your answer and remove the raise line"
expected = (orders_with_returns.groupby('category')['was_returned'].mean() * 100).sort_values(ascending=False)
assert np.allclose(return_rate_by_category.values, expected.values, atol=0.1), "Return rate values don't match"
assert list(return_rate_by_category.index) == list(expected.index), "Wrong order"
print("CORRECT! [return_rate_by_category]")
print(return_rate_by_category.round(1).to_string())


CORRECT! [return_rate_by_category]
category
Home           9.3
Electronics    9.1
Fitness        8.8
Books          5.8
Kitchen        5.7


<details><summary>🔑 Solution</summary>

```python
return_rate_by_category = (
    orders_with_returns.groupby('category')['was_returned'].mean() * 100
).sort_values(ascending=False)
```
</details>

---
# Section 8 — Challenge Round 🔴
*Advanced — multi-step problems, method chaining, `.transform()`*

### Challenge 8.1 — % of category revenue
Add a column `pct_of_category` to `orders`: each order's `final_revenue` as a **percentage of its category's total revenue**.

*Hint: You need `.transform()` here — not `.agg()`. Think about why.*

In [60]:
# YOUR CODE HERE
# raise NotImplementedError()
category_totals = orders.groupby('category')['final_revenue'].transform('sum')
orders['pct_of_category'] = orders['final_revenue'] / category_totals * 100

In [61]:
assert 'final_revenue' in orders.columns, "Solve Exercise 4.2 first -- 'final_revenue' column is missing"
assert 'pct_of_category' in orders.columns, "Column 'pct_of_category' not found -- write your answer and remove the raise line"
cat_sums = orders.groupby('category')['pct_of_category'].sum()
assert np.allclose(cat_sums.values, 100.0, atol=0.1), f"Category percentages don't sum to 100:\n{cat_sums}"
print("CORRECT! [pct_of_category]")
print(orders.groupby('category')['pct_of_category'].sum().round(2))


CORRECT! [pct_of_category]
category
Books          100.0
Electronics    100.0
Fitness        100.0
Home           100.0
Kitchen        100.0
Name: pct_of_category, dtype: float64


<details><summary>💡 Hint</summary>
`.transform('sum')` returns a Series of the same length as `orders`, aligned by group — perfect for computing each row relative to its group total.
</details>

<details><summary>🔑 Solution</summary>

```python
category_totals = orders.groupby('category')['final_revenue'].transform('sum')
orders['pct_of_category'] = orders['final_revenue'] / category_totals * 100
```
</details>

### Challenge 8.2 — Busiest day of the week
Find the **day of the week** (e.g. `"Monday"`) with the **highest total final_revenue** across all orders.  
Store in `busiest_day`.

In [62]:
busiest_day = None
# YOUR CODE HERE
# raise NotImplementedError()
busiest_day = (
    orders.groupby(orders['order_date'].dt.day_name())['final_revenue']
    .sum()
    .idxmax()
)

In [63]:
assert 'final_revenue' in orders.columns, "Solve Exercise 4.2 first -- 'final_revenue' column is missing"
day_revenue = orders.groupby(orders['order_date'].dt.day_name())['final_revenue'].sum()
expected = day_revenue.idxmax()
check(busiest_day, expected, "busiest_day")
print("\nRevenue by day:")
print(day_revenue.sort_values(ascending=False).round(2))


CORRECT! [busiest_day]

Revenue by day:
order_date
Sunday       17712.66
Thursday     17670.38
Saturday     17508.62
Monday       17503.74
Wednesday    17158.64
Tuesday      16453.71
Friday       16277.28
Name: final_revenue, dtype: float64


<details><summary>💡 Hint</summary>
Use `orders['order_date'].dt.day_name()` to get day names, then `groupby` on that.
</details>

<details><summary>🔑 Solution</summary>

```python
busiest_day = (
    orders.groupby(orders['order_date'].dt.day_name())['final_revenue']
    .sum()
    .idxmax()
)
```
</details>

### Challenge 8.3 — Method chaining pipeline
Write a **single chained expression** (no intermediate variables) to produce a DataFrame showing:
- Only **Gold** and **Platinum** tier customers (from `orders_with_tier`)
- Only **Delivered** orders
- Grouped by **loyalty_tier** and **category**
- Showing: `total_revenue` (sum of final_revenue) and `order_count` (count of order_id)
- Sorted by `total_revenue` descending

Store in `vip_summary`.

In [64]:
vip_summary = None
# YOUR CODE HERE
# raise NotImplementedError()
vip_summary = (
    orders_with_tier
    .query("loyalty_tier in ['Gold', 'Platinum'] and status == 'Delivered'")
    .groupby(['loyalty_tier', 'category'])
    .agg(
        total_revenue = ('final_revenue', 'sum'),
        order_count   = ('order_id',      'count')
    )
    .sort_values('total_revenue', ascending=False)
)

In [65]:
try:
    orders_with_tier
except NameError:
    raise AssertionError("Solve Exercise 7.1 first -- run cell 7.1 to initialise orders_with_tier")
assert orders_with_tier is not None, "Solve Exercise 7.1 first -- orders_with_tier is None"
assert 'final_revenue' in orders_with_tier.columns, "Solve Exercise 4.2 first -- 'final_revenue' column is missing in orders_with_tier"
assert isinstance(vip_summary, pd.DataFrame), "vip_summary should be a DataFrame -- write your answer and remove the raise line"
assert set(vip_summary.columns) == {'total_revenue', 'order_count'}, f"Wrong columns: {list(vip_summary.columns)}"
assert set(vip_summary.index.get_level_values('loyalty_tier')) <= {'Gold', 'Platinum'}, "Should only contain Gold and Platinum tiers"
assert vip_summary['total_revenue'].is_monotonic_decreasing, "Should be sorted by total_revenue descending"
print("CORRECT! [vip_summary]")
print(vip_summary.round(2))


CORRECT! [vip_summary]
                          total_revenue  order_count
loyalty_tier category                               
Gold         Fitness            4462.83           45
             Kitchen            3997.90           26
             Electronics        3472.50           25
             Home               2391.22           27
Platinum     Kitchen            2322.64           14
Gold         Books              1906.40           23
Platinum     Electronics        1832.06           17
             Home               1405.30           11
             Fitness            1400.87           15
             Books               626.31            7


<details><summary>🔑 Solution</summary>

```python
vip_summary = (
    orders_with_tier
    .query("loyalty_tier in ['Gold', 'Platinum'] and status == 'Delivered'")
    .groupby(['loyalty_tier', 'category'])
    .agg(
        total_revenue = ('final_revenue', 'sum'),
        order_count   = ('order_id',      'count')
    )
    .sort_values('total_revenue', ascending=False)
)
```
</details>

---
## Final Score
Run this cell to see how many exercises you've completed!

In [66]:
# Run this after completing exercises
# For a fully accurate count: Kernel -> Restart & Run All first

sections = {
    "1. Exploring Data":         5,
    "2. Indexing & Slicing":      4,
    "3. Filtering & Sorting":     5,
    "4. Derived Columns & NaN":   3,
    "5. GroupBy & Aggregations":  5,
    "6. DateTime & String":       4,
    "7. Merging DataFrames":      3,
    "8. Challenge Round":         3,
}
total_exercises = sum(sections.values())
correct   = _score["correct"]
attempted = _score["total"]
pct = round(correct / attempted * 100) if attempted else 0

print("=" * 47)
print("  ShopStream Pandas Practice Lab -- Results")
print("=" * 47)
for s, n in sections.items():
    print(f"  {s:<35} {n:>2} exercises")
print("-" * 47)
print(f"  Total exercises : {total_exercises}")
print(f"  Score           : {correct} / {attempted} checked  ({pct}%)")
print("=" * 47)
print()
if attempted == 0:
    print("  No check() exercises run yet -- get started!")
elif pct == 100:
    print("  All checked exercises correct -- Pandas pro!")
elif pct >= 75:
    print("  Great work! Keep pushing on the challenges.")
elif pct >= 50:
    print("  Solid effort -- revisit the ones you missed.")
else:
    print("  Keep going -- every rep builds the muscle!")
print()
print("  Follow Afnan on LinkedIn for more content:")
print("  linkedin.com/in/afnan-darga")


  ShopStream Pandas Practice Lab -- Results
  1. Exploring Data                    5 exercises
  2. Indexing & Slicing                4 exercises
  3. Filtering & Sorting               5 exercises
  4. Derived Columns & NaN             3 exercises
  5. GroupBy & Aggregations            5 exercises
  6. DateTime & String                 4 exercises
  7. Merging DataFrames                3 exercises
  8. Challenge Round                   3 exercises
-----------------------------------------------
  Total exercises : 32
  Score           : 12 / 12 checked  (100%)

  All checked exercises correct -- Pandas pro!

  Follow Afnan on LinkedIn for more content:
  linkedin.com/in/afnan-darga


---
*Built with ❤️ by Afnan Faiyazahmed Darga*  
*Free to share — tag me if you find it useful!*